# Step 3 — SpatioType assignment on TCGA

Aggregates per-spot ST cluster labels (from step 2) into a single SpatioType label per patient.

**Approach** (from `spatial_clusters/wining.r`):
1. Re-cluster the TCGA per-domain Seurat object with the final "winning" Seurat parameters.
2. For each patient (`tissue_submitter_id`), compute the fraction of their pseudo-spots falling in each cluster — a patient × cluster composition matrix.
3. Run hierarchical clustering (`ward.D2`) on the column-scaled composition matrix, cut at `k=3` → three patient SpatioTypes.
4. Sanity-check against the originally published 3 SpatioTypes (ARI, RI) and against patient survival (KM + log-rank on DFI).

**Inputs (replace paths with your own)**
- `new_tcga.qs` — TCGA per-domain Seurat object with patient metadata (`tissue_submitter_id`, `DFI`, `DFI.time`).
- `orginal_3_spatyotypes.RDS` — the previously published per-patient SpatioType assignment, used only as a comparison reference.

**Outputs**
- `heatmap_clusters_df` — per-patient SpatioType (1/2/3) table.
- KM plot + clustering-metric summary (`rand_index`, `adjusted_rand_index`, log-rank p-value).

## Setup

In [ ]:
library(reticulate)
use_condaenv("tridentenv_v5", required = TRUE)

library(qs)
library(dplyr)
library(tidyr)
library(mclust)
library(aricode)
library(survival)
library(survminer)
library(tibble)
library(ggplot2)
library(data.table)
library(Seurat)
library(sctransform)
library(Matrix)

project <- 'TCGA_BRCA'

## Load TCGA Seurat object and the originally published SpatioTypes

In [ ]:
pred <- qread("/vf/users/Ruppin_ST/Cell_revisions/spatial_clusters/new_tcga.qs")

ori_spatiotypes <- readRDS("/vf/users/Ruppin_ST/Cell_revisions/spatial_clusters/orginal_3_spatyotypes.RDS")
ori_spatiotypes <- ori_spatiotypes[, c('tissue_submitter_id', 'SpatioType')]

## Re-cluster with the final winning Seurat parameters

Note these are different from step 2's parameters: step 2 produces the 11 ST cluster labels (used for biology / markers). Step 3 re-clusters under a parameter set tuned for downstream SpatioType separation.

In [ ]:
norm_method     <- 'CLR'
feat_sel_method <- 'dispersion'
nfeatures       <- 4000
dims_end        <- 30
k_param         <- 3
resolution      <- 0.45
modularity_fxn  <- 1
algorithm       <- 1

pred <- NormalizeData(pred, normalization.method = norm_method)
pred <- FindVariableFeatures(pred, selection.method = feat_sel_method, nfeatures = nfeatures)
pred <- ScaleData(pred)

dims <- 1:dims_end
pred <- RunPCA(pred, npcs = dims_end)
pred <- FindNeighbors(pred, dims = dims, k.param = as.integer(k_param))
pred <- FindClusters(
  pred,
  resolution     = resolution,
  modularity.fxn = modularity_fxn,
  algorithm      = algorithm,
  verbose        = FALSE
)

length(unique(pred$seurat_clusters))

## Per-patient composition matrix

For each patient, compute the fraction of pseudo-spots in each cluster. Result is `(n_patients × n_clusters)`.

In [ ]:
dfc <- cbind(pred@meta.data, data.frame(cell = colnames(pred)))
rel_col <- 'seurat_clusters'

fraction_df <- dfc %>%
  group_by(tissue_submitter_id, !!sym(rel_col)) %>%
  summarise(total_spots = n(), .groups = 'drop') %>%
  group_by(tissue_submitter_id) %>%
  mutate(fraction_spots = total_spots / sum(total_spots))

heatmap_df <- fraction_df %>%
  select(tissue_submitter_id, !!sym(rel_col), fraction_spots) %>%
  pivot_wider(
    names_from  = tissue_submitter_id,
    values_from = fraction_spots,
    values_fill = 0
  )

heatmap_matrix <- heatmap_df %>%
  column_to_rownames(var = rel_col) %>%
  as.matrix() %>%
  t()  # rows = patients, cols = clusters
dim(heatmap_matrix)

## Hierarchical clustering of patients → 3 SpatioTypes

Scale the columns (per-cluster z-score across patients), then `hclust` with `ward.D2` and cut at `k=3`.

In [ ]:
scaled_features <- scale(heatmap_matrix)
linkage_matrix  <- hclust(dist(scaled_features), method = 'ward.D2')
cluster_labels  <- cutree(linkage_matrix, k = 3)

heatmap_clusters_df <- data.frame(
  tissue_submitter_id = rownames(heatmap_matrix),
  SpatioType          = cluster_labels
)
head(heatmap_clusters_df)
table(heatmap_clusters_df$SpatioType)

## Compare to the originally published 3 SpatioTypes

Rand Index / Adjusted Rand Index between our new SpatioType assignment and the previously published one — a high ARI is the sanity check that the pipeline reproduces.

In [ ]:
df_merged <- merge(
  ori_spatiotypes[, c('tissue_submitter_id', 'SpatioType')],
  heatmap_clusters_df,
  by = 'tissue_submitter_id',
  suffixes = c('_ori', '_heatmap')
)

ari <- adjustedRandIndex(df_merged$SpatioType_ori, df_merged$SpatioType_heatmap)
ri  <- RI(df_merged$SpatioType_ori, df_merged$SpatioType_heatmap)
cat('RI=', ri, '  ARI=', ari, '\n')

## Survival check (DFI)

Kaplan–Meier on DFI stratified by the new SpatioType; log-rank for the global test.

In [ ]:
surv_df <- distinct(dfc[, c('tissue_submitter_id', 'DFI', 'DFI.time')]) %>%
  inner_join(heatmap_clusters_df, by = 'tissue_submitter_id')

surv_obj     <- Surv(time = surv_df$DFI.time, event = surv_df$DFI)
km_fit       <- survfit(surv_obj ~ SpatioType, data = surv_df)
logrank_test <- survdiff(surv_obj ~ SpatioType, data = surv_df)
p_value      <- 1 - pchisq(logrank_test$chisq, length(logrank_test$n) - 1)

output_metrics <- data.frame(
  scenario            = rel_col,
  clusters            = length(unique(dfc[, rel_col])),
  rand_index          = ri,
  adjusted_rand_index = ari,
  logrank_p_value     = p_value
)
output_metrics

In [ ]:
options(repr.plot.width = 5, repr.plot.height = 5)
print(ggsurvplot(km_fit, data = surv_df))

## (Optional) Contingency + composition heatmaps

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 5)

tab_df <- as.data.frame(table(df_merged$SpatioType_ori, df_merged$SpatioType_heatmap))
colnames(tab_df) <- c('Original', 'Heatmap', 'Count')
tab_df <- tab_df[tab_df$Original %in% c('Proliferation-Enriched', 'Immune-Inactive', 'Immune-Modulated'), ]

ggplot(tab_df, aes(x = Heatmap, y = Original, fill = Count)) +
  geom_tile() +
  geom_text(aes(label = Count), color = 'black') +
  scale_fill_gradient(low = 'gray90', high = 'red') +
  theme_minimal() +
  labs(title = paste('Cluster Agreement -', rel_col), x = 'Heatmap Cluster', y = 'Original Cluster')

In [ ]:
library(pheatmap)
options(repr.plot.width = 14, repr.plot.height = 5)

pheatmap(
  t(heatmap_matrix),
  scale              = 'none',
  clustering_method  = 'ward.D',
  cluster_rows       = FALSE,
  cluster_cols       = TRUE,
  cutree_cols        = 3,
  fontsize_col       = 0.1,
  width              = 15,
  height             = 12
)